## Browser Extension을 사용하는 AgentCore Browser Tool

이 예제에서는 AgentCore Browser에서 [browser extension](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-extensions.html)을 사용하는 방법을 알아봅니다.

Browser Extension을 사용하면 세션을 생성할 때 사용자 지정 extension을 브라우저 세션에 설치할 수 있습니다. 이를 통해 자동화 작업, web scraping, 테스트 등에 맞게 자체 extension으로 브라우저 동작을 사용자 지정할 수 있습니다.

In [ ]:
!pip install -qU -r requirements.txt

전역 변수 선언

In [ ]:
import boto3
import json
import sys
from botocore.exceptions import ClientError

sys.path.append("../helpers/")

iam_boto3 = boto3.client("iam")
s3 = boto3.client("s3")
browser_boto3 = boto3.client("bedrock-agentcore-control")
browser_cli = boto3.client("bedrock-agentcore")

session = boto3.Session()
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
REGION = session.region_name

BROWSER_NAME = "browser_with_extensions"
BUCKET_NAME = f"ac-browser-demos-{ACCOUNT_ID}-{REGION}"
AC_ROLE_NAME = "ac-browser-ext-execution-role"

### 1. Playwright를 사용한 로컬 테스트

`extension` 폴더에서 미리 만들어진 extension을 확인할 수 있습니다.

이 단계에서는 로컬 세션을 시작하고 Playwright로 extension이 작동하는지 테스트합니다.

브라우저가 시작되면 Chrome에서 extension을 클릭해 로컬에서 작동하는 모습을 확인하세요.

![local_extension.png](img/local_extension.png)

In [ ]:
from playwright.async_api import async_playwright

extension_path = "./extension"

async with async_playwright() as p:
    context = await p.chromium.launch_persistent_context(
        user_data_dir="./user-data",
        headless=False,
        args=[
            f"--disable-extensions-except={extension_path}",
            f"--load-extension={extension_path}",
        ],
    )

    page = await context.new_page()
    await page.goto("chrome://extensions/")
    await page.wait_for_timeout(2000)

    input("Press Enter to close...")
    await context.close()

#### 1.1 S3 Bucket 생성

나중에 다운로드할 브라우저 녹화를 저장할 S3 Bucket이 없다면 새로 생성해야 합니다.

In [ ]:
try:
    # Bucket이 있는지 확인
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket {BUCKET_NAME} already exists")
except ClientError:
    # Bucket 생성
    create_params = {"Bucket": BUCKET_NAME}
    if REGION != "us-east-1":
        create_params["CreateBucketConfiguration"] = {"LocationConstraint": REGION}
    s3.create_bucket(**create_params)
    print(f"Bucket {BUCKET_NAME} created in {REGION}")

#### 1.2 IAM role 생성

그런 다음 AgentCore Browser에 연결할 사용자 지정 IAM role을 생성합니다.

In [ ]:
try:
    # Trust policy 정의
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }

    # Role 생성
    browser_role = iam_boto3.create_role(RoleName=AC_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))

    browser_role_arn = browser_role["Role"]["Arn"]

    print(f"Role ARN: {browser_role_arn}")

    # 녹화를 위한 S3 policy
    ac_browser_policies = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "s3:PutObject",
                    "s3:GetObject",
                    "s3:GetObjectVersion",
                    "s3:ListBucket",
                    "s3:ListMultipartUploadParts",
                    "s3:AbortMultipartUpload",
                ],
                "Resource": [
                    f"arn:aws:s3:::{BUCKET_NAME}",
                    f"arn:aws:s3:::{BUCKET_NAME}/*",
                ],
            }
        ],
    }

    # S3 inline policy 추가
    iam_boto3.put_role_policy(
        RoleName=AC_ROLE_NAME,
        PolicyName="ac_custom_policies",
        PolicyDocument=json.dumps(ac_browser_policies),
    )

    # Bedrock managed policy 연결
    iam_boto3.attach_role_policy(
        RoleName=AC_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/AmazonBedrockFullAccess",
    )

except ClientError as e:
    print(f"Exception: {e}")
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        browser_role_arn = iam_boto3.get_role(RoleName=AC_ROLE_NAME)["Role"]["Arn"]
        print(f"Arn captured: {browser_role_arn}")

IAM role이 전파되도록 10초 동안 기다립니다.

In [ ]:
import time

time.sleep(10)

#### 1.3 사용자 지정 AgentCore Browser 생성

Extension을 zip file로 압축해 S3 Bucket에 업로드합니다.

In [ ]:
![ -f sample-extension.zip ] && rm sample-extension.zip
!cd extension && zip -r ../sample_extension.zip .
!cd ..

S3에 업로드

In [ ]:
s3.upload_file(
    "sample_extension.zip",
    BUCKET_NAME,
    "extensions/sample_extension.zip",
    ExtraArgs={"ContentType": "application/zip"},
)

이 예제에서는 사용자 지정 브라우저를 생성하지만 이 기능은 managed browser(`aws.browser.v1`)에서도 작동합니다.

In [ ]:
created_browser = browser_boto3.create_browser(
    name=BROWSER_NAME,
    executionRoleArn=browser_role_arn,
    networkConfiguration={"networkMode": "PUBLIC"},
    recording={
        "enabled": True,
        "s3Location": {"bucket": BUCKET_NAME, "prefix": "browser_recordings/"},
    },
)

browser_id = created_browser["browserId"]
print(f"Browser ID: {browser_id}")

### 2. 테스트

테스트를 시작하기 위해 새 브라우저 세션을 시작합니다.

In [ ]:
response = browser_cli.start_browser_session(
    browserIdentifier=browser_id,
    extensions=[
        {
            "location": {
                "s3": {
                    "bucket": BUCKET_NAME,
                    "prefix": "extensions/sample_extension.zip",
                }
            }
        }
    ],
)

session_id = response["sessionId"]
print(f"Session ID: {session_id}")

다음 셀에서는 IAM 자격 증명을 추가하기 위해 SigV4로 request에 서명합니다.

In [ ]:
import browser_helper as helper

url = helper.get_url(browser_id, session_id)
headers = helper.get_signed_headers(url)
headers

#### 2.1 AgentCore Browser에서 테스트

이제 Playwright를 사용해 extension을 확인하고 테스트합니다.
[Playwright](https://playwright.dev/docs/intro)는 AgentCore Browser에서 지원하는 Web Testing and Automation framework입니다.

Playwright 코드를 실행하기 전에 AWS Console의 Browser로 이동해 *View live session* 버튼을 클릭하세요.

![browser_console.png](img/browser_console.png)

In [ ]:
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(url, headers=headers)
    page = browser.contexts[0].pages[0] if browser.contexts else await browser.new_context().new_page()

    await page.goto("chrome://extensions/")
    await page.wait_for_timeout(2000)

코드 실행 후 브라우저를 열고 extension을 클릭해 AgentCore Browser에서 작동하는 모습을 확인할 수 있습니다.

![remote_extension.png](img/remote_extension.png)

#### 2.3 세션 중지

세션을 중지합니다.

In [ ]:
stoped_session = browser_cli.stop_browser_session(browserIdentifier=browser_id, sessionId=session_id)
stoped_session

### 3. 정리(선택 사항)

사용자 지정 AgentCore Browser와 Profile을 삭제합니다.

In [ ]:
browser_boto3.delete_browser(browserId=browser_id)